[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/03_%E7%B5%B1%E8%A8%88_3%E7%B4%9A/01_%E5%88%86%E6%95%A3%E3%83%BB%E6%A8%99%E6%BA%96%E5%81%8F%E5%B7%AE%E3%83%BB%E6%A8%99%E6%BA%96%E5%8C%96.ipynb)

> Colabで実行可。最初に「① セットアップ」セルを実行（Colabはデータを自動生成、ローカルは何もしない）。

In [ ]:
# === ① セットアップ（最初に実行してください）===============================
# Colabなどデータが無い環境では、教材と同一の合成データを自動生成します。
# ローカル/Codespacesで data/ がある場合は何もしません（再現性のため乱数seedは固定）。
import os
if not os.path.exists('../data/students_scores.csv'):   # データが無い環境（Colab等）か判定
    os.makedirs('/content/nb', exist_ok=True); os.chdir('/content/nb')  # ../data を使えるよう作業場所を作る
    os.makedirs('../data', exist_ok=True)               # データ保存先フォルダを作る
    import numpy as np, pandas as pd                     # 数値計算と表データのライブラリ
    D = '../data'; rng = np.random.default_rng(42)       # 保存先と、結果を固定する乱数生成器
    # --- 生徒120人の成績データを作る ---
    n = 120; cls = rng.choice(['A','B','C'], n)          # 人数とクラス
    math = np.clip(rng.normal(65,15,n),0,100).round().astype(int)        # 数学（0〜100点）
    eng = np.clip(0.6*math+rng.normal(28,10,n),0,100).round().astype(int) # 英語（数学と相関）
    jp = np.clip(rng.normal(70,12,n),0,100).round().astype(int)          # 国語
    hrs = np.clip(rng.normal(1.5,0.8,n)+math/100,0,None).round(1)        # 勉強時間
    pd.DataFrame({'生徒ID':[f'S{i:03d}' for i in range(1,n+1)],'クラス':cls,
        '数学':math,'英語':eng,'国語':jp,'勉強時間':hrs}).to_csv(f'{D}/students_scores.csv',index=False,encoding='utf-8-sig')
    # --- 30日間の天気データ ---
    days = pd.date_range('2026-06-01', periods=30)       # 6月の30日分の日付
    temp = (24+4*np.sin(np.arange(30)/5)+rng.normal(0,2,30)).round(1)    # 気温
    rain = np.clip(rng.gamma(1.2,4,30)*(rng.random(30)<0.4),0,None).round(1)  # 降水量
    pd.DataFrame({'日付':days.strftime('%Y-%m-%d'),'気温':temp,'降水量':rain}).to_csv(f'{D}/weather.csv',index=False,encoding='utf-8-sig')
    # --- BtoB商談400件 ---
    m = 400; inds = rng.choice(['製造','IT','小売','医療','金融','建設'],m,p=[.25,.2,.18,.12,.15,.1])  # 業界
    chs = ['展示会','Web広告','メルマガ','紹介','テレアポ']; ch = rng.choice(chs,m,p=[.2,.3,.2,.15,.15])  # 獲得チャネル
    deal = (rng.lognormal(13,0.7,m)).round(-3).astype(int)   # 商談金額
    wr = {'展示会':.35,'Web広告':.18,'メルマガ':.12,'紹介':.45,'テレアポ':.15}  # チャネル別の受注率
    won = np.array([rng.random()<wr[c] for c in ch])         # 受注したか（チャネルで確率を変える）
    dt = pd.to_datetime('2026-01-01')+pd.to_timedelta(rng.integers(0,180,m),unit='D')  # 受注日
    pd.DataFrame({'商談ID':[f'D{i:04d}' for i in range(1,m+1)],'受注日':dt.strftime('%Y-%m-%d'),
        '業界':inds,'獲得チャネル':ch,'商談金額':deal,'担当者':rng.choice(['田中','鈴木','佐藤','高橋'],m),
        '受注':won.astype(int)}).to_csv(f'{D}/sales_btob.csv',index=False,encoding='utf-8-sig')
    # --- 月×チャネルのマーケ実績 ---
    months = pd.date_range('2026-01-01',periods=6,freq='MS').strftime('%Y-%m')  # 6か月分
    base = {'展示会':2000,'Web広告':12000,'メルマガ':6000,'紹介':800,'テレアポ':1500}  # 表示回数の目安
    ctr = {'展示会':.08,'Web広告':.03,'メルマガ':.05,'紹介':.2,'テレアポ':.12}   # クリック率
    cvr = {'展示会':.12,'Web広告':.04,'メルマガ':.06,'紹介':.25,'テレアポ':.09}   # 獲得率
    cpc = {'展示会':30,'Web広告':80,'メルマガ':5,'紹介':0,'テレアポ':200}        # クリック単価
    rows = []
    for mo in months:                                    # 月ごと
        for c in chs:                                    # チャネルごとに実績を作る
            imp = int(base[c]*rng.uniform(0.8,1.3)); clk = int(imp*ctr[c]*rng.uniform(0.8,1.2))
            cv = int(clk*cvr[c]*rng.uniform(0.7,1.3)); cost = int(clk*cpc[c]*rng.uniform(0.9,1.1))
            rows.append([mo,c,imp,clk,cv,cost])
    pd.DataFrame(rows,columns=['月','チャネル','表示回数','クリック数','獲得数','費用']).to_csv(f'{D}/web_marketing.csv',index=False,encoding='utf-8-sig')
    # --- A/Bテストの申込ログ ---
    ab = []
    for grp,p,size in [('A_青ボタン',0.082,2400),('B_緑ボタン',0.104,2380)]:  # 2群の申込率と人数
        for c in (rng.random(size)<p): ab.append([grp,int(c)])   # 各訪問が申込んだか
    pd.DataFrame(ab,columns=['グループ','申込']).sample(frac=1,random_state=1).reset_index(drop=True).to_csv(f'{D}/ab_test.csv',index=False,encoding='utf-8-sig')
    print('✅ 合成データを生成しました（Colab環境）')
else:
    print('✅ data/ を検出しました（ローカル/Codespaces）')

# 統計3級-01. 分散・標準偏差・標準化

**この章でできるようになること**
- ばらつき（散らばり）を1つの数で表せる（分散・標準偏差）
- *なぜ2乗するのか*・*なぜ√で戻すのか* を自分の言葉で説明できる
- `÷n`（母分散）と `÷(n-1)`（不偏分散）の違いがわかる
- 標準化（z得点）・偏差値で、条件のちがうデータを公平に比べられる

**前提**：`01_Python基礎/05〜07`（NumPy・pandas・matplotlib）と統計4級の代表値　／　**所要**：約30分　／　**レベル**：統計検定3級相当

In [ ]:
import numpy as np               # 数値計算
import pandas as pd              # 表データ
import matplotlib.pyplot as plt  # グラフ描画
from scipy import stats          # 統計関数（分布・検定など）
plt.rcParams['axes.unicode_minus'] = False       # マイナス記号の文字化けを防ぐ
df = pd.read_csv('../data/students_scores.csv')   # 生徒データを読み込む
df.head()                        # 先頭5行を確認

### 使うデータ：`students_scores.csv`（架空の生徒120人の成績）
1行＝生徒1人。数学・英語・国語は0〜100点、勉強時間は1日あたりの時間です。

| 生徒ID | クラス | 数学 | 英語 | 国語 | 勉強時間 |
|---|---|---|---|---|---|
| S001 | A | 52 | 58 | 49 | 2.0 |
| S002 | C | 80 | 79 | 74 | 3.4 |
| S003 | B | 40 | 65 | 91 | 2.0 |

（下のセルの `df.head()` で先頭5行を実際に確認できます）

## 0. なぜ「ばらつき」を測るの？（まず直感）

2つのクラスのテスト。**平均はどちらも60点**。でも中身はまるで違います。

In [ ]:
A = np.array([58, 60, 62, 59, 61])   # 平均60・まとまっている
B = np.array([30, 90, 60, 40, 80])   # 平均60・散らばっている
print('平均: A =', A.mean(), ' B =', B.mean(), '（同じ！）')

fig, ax = plt.subplots(1, 2, figsize=(9, 3), sharey=True)   # 2つ並べて描く
ax[0].stem(A); ax[0].axhline(60, color='red', ls='--'); ax[0].set_title('Aクラス（まとまり）')
ax[1].stem(B); ax[1].axhline(60, color='red', ls='--'); ax[1].set_title('Bクラス（散らばり）')
plt.show()

平均が同じでも散らばりは全然ちがう。**この「散らばり具合」を1つの数で表す**のが分散・標準偏差です。

## 1. 偏差 → 分散 → 標準偏差（順番に・なぜ付き）

1. **偏差** = 各データ − 平均（平均からどれだけズレているか）
2. 偏差をそのまま平均すると…？ → **必ず 0**（＋と−が打ち消す）。だから **2乗** して符号を消す。
3. **分散** = 偏差²の平均。ただし単位が「点²」になり直感的でない。
4. **標準偏差** = √分散。単位が元（点）に戻り「平均から平均的に何点ズレるか」と読める。

> **よくある間違い**：「偏差の平均」でばらつきを測ろうとする → 必ず0になり使えない。

In [ ]:
x = np.array([60, 70, 80, 90, 100])
dev = x - x.mean()                       # 偏差
print('偏差:', dev)
print('偏差の合計:', dev.sum(), '← いつも0（だから2乗する）')
var = (dev**2).mean()                    # 分散 = 偏差²の平均
print('分散:', var)
print('標準偏差:', np.sqrt(var).round(2), '← √で単位が元に戻る')

## 2. シミュレーションで体感する

標準偏差を変えてデータを作り、ヒストグラムの広がりを見比べます。

In [ ]:
rng = np.random.default_rng(0)
for sd in [5, 15, 30]:                          # 標準偏差を3通り
    d = rng.normal(60, sd, 1000)                # 平均60・標準偏差sd の1000個
    print(f'設定 std={sd:2d} → 実際の標準偏差={d.std():.1f}')
    plt.hist(d, bins=30, alpha=0.5, label=f'std={sd}')
plt.legend(); plt.title('標準偏差が大きいほど横に広がる'); plt.show()

**標準偏差が大きい＝ヒストグラムが横に広い＝散らばっている**、と体で覚えましょう。

## 3. `÷n` か `÷(n-1)` か（つまずきポイント）

- **`÷n`（母分散）**：手元のデータが“全体（母集団）”そのもののとき。
- **`÷(n-1)`（不偏分散）**：標本から“全体のばらつき”を推定するとき。標本だと小さめに出やすいので n-1 で補正。

> **事故が多い所**：pandas `.var()`/`.std()` は既定で **÷(n-1)**、NumPy `np.var`/`np.std` は **÷n**。同じデータでも値がズレます。

In [ ]:
s = pd.Series([60, 70, 80, 90, 100])
print('pandas s.var()   :', s.var(), '  （÷(n-1) 不偏）')
print('numpy  np.var(s) :', np.var(s), '  （÷n 母分散）')
print('numpy  ddof=1    :', np.var(s, ddof=1), '  （pandasに合わせる）')

## 4. 標準化（z得点）— 条件のちがうデータを公平に比べる

数学（平均65・σ15）で80点 と 英語（平均70・σ8）で80点、**どちらがすごい？**
「平均から標準偏差いくつ分はなれているか」で測るのが **z得点**：

$$ z = \frac{x - 平均}{標準偏差} $$

In [ ]:
for col in ['数学', '英語']:
    z = (80 - df[col].mean()) / df[col].std()    # 80点のz得点
    print(f'{col}で80点 → z = {z:.2f}')
print('zが大きい科目のほうが、その中で相対的に上位')

標準化すると平均0・標準偏差1になり、**単位や難易度がちがっても比較**できます。

## 5. 偏差値

z得点を見やすくしたものが偏差値。**偏差値 = 50 + 10 × z**（平均50・σ10）。

> 偏差値で「上位何%」が決まるのは**分布が正規分布に近いとき**の目安（偏差値70 ≈ 上位約2%）。分布が大きく偏ると崩れます。

In [ ]:
df['数学偏差値'] = 50 + 10 * (df['数学'] - df['数学'].mean()) / df['数学'].std()
print('偏差値の平均と標準偏差:', df['数学偏差値'].mean().round(1), '/', df['数学偏差値'].std().round(1))
df[['生徒ID', '数学', '数学偏差値']].sort_values('数学', ascending=False).head()

## 6. 定数を足す・かけると、分散とSDはどう変わる？

データ全体に **同じ数を足す** と散らばりは変わりません（位置がずれるだけ）。一方 **同じ数をかける** と、標準偏差は |倍率| 倍、分散はその2乗倍になります。

In [ ]:
v = np.array([2, 4, 6, 8, 10])
print('元   : 分散(母,÷n) =', v.var(), ' / SD =', round(v.std(), 3))
print('+10  : 分散 =', (v+10).var(), ' / SD =', round((v+10).std(), 3), '→ 変わらない（位置だけ移動）')
print('×2   : 分散 =', (v*2).var(), ' / SD =', round((v*2).std(), 3), '→ 分散は4倍, SDは2倍')

> 偏差値は `z × 10 + 50`（zスコアを ×10 して +50）という一次変換です。だからどんなテストでも平均50・標準偏差10にそろい、別の試験どうしを公平に比べられます。

## 確認テスト（自動採点）

`ans = None` を**自分の計算で置き換えて実行**すると、その場で正誤が出ます（答えは表示されません）。

In [ ]:
# まず採点用の関数を実行（答え合わせに使うだけ）
def _check(name, got, want, tol=1e-2):
    if got is None:
        print(f'⬜ {name}: まだ入力されていません（ans を埋めてね）'); return
    ok = abs(got - want) <= tol
    print(('✅ 正解！ ' if ok else '❌ もう一度 ') + f'{name}: あなたの答え = {got}')

In [ ]:
# Q1: y = [2,4,6,8,10] の「分散（÷n, 母分散）」を計算して ans に入れよう
y = np.array([2, 4, 6, 8, 10])
ans = None     # 例: np.var(y)
_check('Q1 分散(÷n)', ans, np.var(y))

In [ ]:
# Q2: 同じ y の「不偏分散（÷(n-1)）」を ans に入れよう
ans = None     # ヒント: np.var(y, ddof=1) でもOK
_check('Q2 不偏分散', ans, np.var(y, ddof=1))

In [ ]:
# Q3: 平均50・標準偏差10のテストで「70点」の z得点 を ans に入れよう
ans = None     # ヒント: (70 - 50) / 10
_check('Q3 z得点', ans, (70 - 50) / 10)

In [ ]:
import numpy as np
w = np.array([3, 5, 7, 9, 11])
# Q4: w を「×3」したときの標準偏差(母, ÷n)を ans に（元のSDの3倍になるはず）
ans = None   # 例: (w*3).std()
_check('Q4 ×3後のSD', ans, (w*3).std())

---
## 練習問題

**問1.** `国語` の **不偏標準偏差** と **変動係数（標準偏差÷平均）** を求めよう。

**問2.** `英語` を標準化した列を作り、平均≈0・標準偏差≈1 になることを確かめよう。

**問3.** ある生徒は数学・英語ともに **75点**。偏差値が高いのはどちらの教科？ 理由も。

**問4.（記述）** なぜ「分散そのもの」ではなく「分散の√＝標準偏差」を使うのか、*単位*の観点で説明しよう。

**問5.** あるテストの平均46点・標準偏差9点だった。全員に一律 +5点 すると平均と標準偏差はどうなる？さらに全員の点を 1.1倍 すると標準偏差は何倍になる？ コードで確かめよう。

In [ ]:
# 問1〜3（問4は記述）


> **解答例（解説つき）は別ページ**（ネタバレ防止）。
> 自分で解いてから **[01_分散・標準偏差・標準化 の解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/03_%E7%B5%B1%E8%A8%88_3%E7%B4%9A/01_%E5%88%86%E6%95%A3%E3%83%BB%E6%A8%99%E6%BA%96%E5%81%8F%E5%B7%AE%E3%83%BB%E6%A8%99%E6%BA%96%E5%8C%96.md)**

## 用語集 ＆ チートシート

| 用語 | 意味 | 式 |
|---|---|---|
| 偏差 | 各データ − 平均 | $x_i - \bar{x}$ |
| 分散（母） | 偏差²の平均（÷n） | $\frac{1}{n}\sum (x_i-\bar{x})^2$ |
| 不偏分散 | ÷(n-1) の分散 | $\frac{1}{n-1}\sum (x_i-\bar{x})^2$ |
| 標準偏差 | √分散（単位が元に戻る） | $\sqrt{分散}$ |
| 標準化(z) | 平均0・σ1にそろえる | $z=\frac{x-\bar{x}}{s}$ |
| 偏差値 | zを見やすく | $50+10z$ |
| 変動係数 | 相対的なばらつき | $\frac{s}{\bar{x}}$ |

In [ ]:
# ── コピペ用チートシート ──
s = df['数学']
s.mean(); s.median()           # 平均・中央値
s.var()                        # 不偏分散 ÷(n-1)
s.std()                        # 不偏標準偏差
np.var(s); np.std(s)           # 母分散・母標準偏差 ÷n
(s - s.mean()) / s.std()       # 標準化（z得点）
50 + 10 * (s - s.mean()) / s.std()   # 偏差値
s.std() / s.mean()             # 変動係数

## 完了ログ

このノートを終えたら、下のセルに **名前** を入れて実行してください（学習の記録用）。
記録用URL（`LOG_URL`）は配布時に設定されます。空欄のままなら記録されず、メッセージが出るだけです。

In [ ]:
# === 完了ログ ===  このノートを終えたら、NAME を入れて実行してください。
import datetime
try:
    import requests
except ImportError:
    requests = None

NAME = ""       # ← あなたの名前（例: 山田）
LOG_URL = ""    # ← 記録用URL（配布時に設定）
LOG_TOKEN = ""  # ← 記録用トークン（配布時に設定）
NOTEBOOK = "03_統計_3級/01_分散・標準偏差・標準化"

if NAME and LOG_URL and requests:
    try:
        requests.post(LOG_URL, json={"token": LOG_TOKEN, "name": NAME, "notebook": NOTEBOOK,
                      "time": datetime.datetime.now().isoformat(timespec="seconds")}, timeout=10)
        print(f"記録しました: {NAME} / {NOTEBOOK}")
    except Exception as e:
        print("記録に失敗しました（URL/ネットワークを確認）:", e)
else:
    print(f"{NOTEBOOK}: NAME と LOG_URL を設定すると、学習の完了が記録されます。")